# OnTrack quickstart (Google Colab, T4 GPU)

End-to-end: install → build knowledge-base index → train the triage classifier → load MedGemma 1.5 4B (float16 on a T4, or 4-bit on a smaller GPU) → chat → run the RAG vs no-RAG evaluation on the 60-item held-out test set → export the human-scoring CSV.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Open https://huggingface.co/google/medgemma-1.5-4b-it while logged in and accept the Health AI Developer Foundations terms (access is gated).
3. Create a Hugging Face *read* token and save it in Colab **Secrets** (key icon on the left) as `HF_TOKEN`, with notebook access on.

OnTrack is an education prototype. It does not diagnose and never gives medication or insulin doses.


## 1. Get the code

In [ ]:
import os
# Before torch is imported (next cell). Reduces fragmentation on the T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO = "https://github.com/Gen3ratorX/ontrack-food.git"
BRANCH = "main"
if not os.path.exists("/content/ontrack-food"):
    !git clone -b {BRANCH} {REPO} /content/ontrack-food
# If the branch is not pushed yet: upload ontrack-food.zip to /content and run
#   !unzip -q /content/ontrack-food.zip -d /content
%cd /content/ontrack-food


In [ ]:
!pip -q install -r requirements-colab.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Hugging Face login (needed for MedGemma and AfriMed-QA)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Colab secret found, falling back to interactive login:", e)
    login()

## 3. Knowledge base and retrieval (CPU is fine)

In [ ]:
!python scripts/build_wafct_extract.py      # committed extract is reused; FAO download only with --force
!python scripts/build_index.py
!python -m ontrack.eval.retrieval_eval --k 5 | head -3

In [ ]:
from ontrack.rag.retriever import Retriever
retriever = Retriever.from_config()
for h in retriever.search("Is kenkey better than banku for blood sugar?"):
    print(f"{h.score:.3f}", h.chunk_id, "|", h.title)

## 4. Triage classifier (TF-IDF + logistic regression) and the safety gate

In [ ]:
!python -m ontrack.classifier.baseline
from ontrack.classifier.gate import SafetyGate
gate = SafetyGate.from_config()
for q in ["Is waakye okay for breakfast?", "How many units of insulin for fufu?", "My father fainted and is not responding"]:
    print(gate.classify(q), "<-", q)

## 4b. Meal / food glycaemic-impact classifier (CPU, no model needed)
Recognises Ghanaian and West African foods in free text, including local names (dɔkono, bayerɛ, nkontommire…) and typos, then rates the expected blood-sugar impact as **low / medium / high / insufficient_evidence**:
- An **estimated glycaemic load** is given only when a food has both a cited GI (Ghanaian GI studies) and a WAFCT 2019 carbohydrate value.
- Otherwise the rating uses carbohydrate only or the GI class only, and says so.
- Protein, vegetables, legumes and fat from at least two foods lower the rating by one level.
- Every result lists the foods that drove it and its sources.

All numbers are estimates based on assumed portions. The TF-IDF + LR model is a learned comparison, trained on synthetic engine labels, so its CV score shows label leakage.

In [ ]:
from ontrack.meal.engine import analyse_meal
for meal in ["2 balls of kenkey with fried fish and shito",
             "half ball of banku with okro stew and grilled tilapia",
             "jollof rice with fried chicken, salad and a bottle of coke",
             "kokonte and groundnut soup"]:
    print(analyse_meal(meal).to_context(), "\n")

In [ ]:
# Learned comparison model + evaluation (synthetic data; see README for the leakage caveat)
!python scripts/make_meal_dataset.py
!python -m ontrack.meal.learned | tail -60
!python -m ontrack.eval.meal_eval

## 5. Load MedGemma 1.5 4B

Float16 weights are about 8.6 GB, so they fit in a 15 GB T4 and run faster than bitsandbytes 4-bit. This cell uses float16 when GPU memory is at least 14 GB, and 4-bit otherwise. Set `FORCE_4BIT = True` to force 4-bit.


In [ ]:
import torch
from ontrack.config import CONFIG
from ontrack.rag.generator import MedGemmaGenerator

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
FORCE_4BIT = False  # set True to force bitsandbytes 4-bit (slower, uses less memory)
if FORCE_4BIT:
    use_4bit = True
elif mem_gb >= 14:
    use_4bit = False
elif mem_gb > 0:
    use_4bit = True
else:
    use_4bit = False
print(f"GPU memory {mem_gb:.1f} GB -> {'4-bit NF4' if use_4bit else 'float16'}")
gen = MedGemmaGenerator(CONFIG.model_id, load_in_4bit=use_4bit, max_new_tokens=256)
print(gen.model_id, gen.torch_dtype, "batch_size", gen.batch_size)


## 6. Chat: RAG vs no-RAG

In [ ]:
from ontrack.pipeline import OnTrackAssistant
rag_bot = OnTrackAssistant(gen, retriever, gate, use_rag=True)
base_bot = OnTrackAssistant(gen, None, gate, use_rag=False)
q = "I have type 2 diabetes and I'm about to eat jollof rice with fried chicken and salad. Is this a good meal choice?"
for bot in (rag_bot, base_bot):
    r = bot.answer(q)
    print(f"=== {r.mode} (triage={r.triage_label})\n{r.answer}\n", [s['doc_id'] for s in r.sources])

In [ ]:
print(rag_bot.answer("I'm shaking and sweating, my sugar is 2.6").answer)

In [ ]:
# A meal question: the safety gate runs first, then the meal analysis [M] is added to the RAG context
r = rag_bot.answer("I'm about to eat 2 balls of kenkey with fried fish and shito. How will it affect my sugar?")
print(r.answer); print(r.meal_analysis["label"], r.meal_analysis["drivers"])

## 7. Optional: zero-shot MedGemma triage vs the TF-IDF baseline

In [ ]:
import json
from ontrack.classifier.zero_shot import evaluate_zero_shot
rows = [json.loads(l) for l in open(CONFIG.triage_data)]
rep = evaluate_zero_shot(gen, rows)
print({k: round(v["recall"], 3) for k, v in rep.items() if isinstance(v, dict) and "recall" in v}, "accuracy", round(rep["accuracy"], 3))

## 8. Evaluation

Three steps, slowest last. Generation is batched (float16 on a T4 starts at 4, and drops to 2 for long prompts or on CUDA OOM), emergency and clinician answers are fixed templates and are not generated, and each finished row is appended to `responses.jsonl` so a disconnected Colab can resume by rerunning the same cell.

1. **Quick eval** — 20 held-out items, stratified by triage label and culture (seed 42), both modes, no judge. The cell prints a rough time estimate, then elapsed time and an ETA.
2. **Optional full 60** — the whole frozen test split, still without a judge.
3. **Optional judge-only** — a later batched MedGemma pass over the saved `responses.jsonl`.

If Google Drive is mounted, results go to `MyDrive/ontrack-food/runs/` instead of the local disk.


In [ ]:
# Quick eval: 20 items, rag + no_rag, no judge. Rerun this cell to resume.
from pathlib import Path
from collections import Counter
from ontrack.config import CONFIG
from ontrack.eval.harness import load_jsonl, run, stratified_subset

drive = Path("/content/drive/MyDrive")
quick_out = (drive / "ontrack-food/runs/quick") if drive.is_dir() else Path("artifacts/runs/quick")
print("Saving to", quick_out, "(Drive is mounted)" if drive.is_dir() else "(local only; mount Drive to keep results across disconnects)")

quick_items = stratified_subset(load_jsonl(CONFIG.splits_dir / "test.jsonl"), 20, seed=42)
print("Quick subset", len(quick_items),
      "triage", dict(Counter(r["expected_triage"] for r in quick_items)),
      "culture", dict(Counter(r["culture_tag"] for r in quick_items)))
# Equivalent CLI, if you would rather reload the model:
#   python -m ontrack.eval.harness --quick --generator medgemma --fp16 --drive
quick_summary = run(quick_items, ["rag", "no_rag"], gen, retriever, gate, None, quick_out,
                    max_new_tokens=256, batch_size=gen.batch_size)


In [ ]:
# Optional: all 60 held-out items, both modes, no judge. Resume-safe.
from pathlib import Path
from ontrack.config import CONFIG
from ontrack.eval.harness import load_jsonl, run

drive = Path("/content/drive/MyDrive")
full_out = (drive / "ontrack-food/runs/full") if drive.is_dir() else Path("artifacts/runs/full")
print("Saving to", full_out)
full_items = load_jsonl(CONFIG.splits_dir / "test.jsonl")
# Equivalent CLI:
#   python -m ontrack.eval.harness --generator medgemma --fp16 --max-new-tokens 256 --judge none --drive --out full
full_summary = run(full_items, ["rag", "no_rag"], gen, retriever, gate, None, full_out,
                   max_new_tokens=256, batch_size=gen.batch_size)


In [ ]:
# Optional judge-only pass over saved responses (batched). Does not regenerate answers.
from pathlib import Path
from ontrack.eval.harness import judge_saved
from ontrack.eval.judge import LLMJudge
import pandas as pd

candidates = []
for name in ("full", "quick"):
    for base in (Path("/content/drive/MyDrive/ontrack-food/runs"), Path("artifacts/runs")):
        candidates.append(base / name / "responses.jsonl")
judge_path = next(p for p in candidates if p.exists())
print("Judging", judge_path)
# Equivalent CLI:
#   python -m ontrack.eval.harness --judge-only <responses.jsonl> --judge llm --fp16 --batch-size 8
judge_summary = judge_saved(judge_path, LLMJudge(gen), batch_size=gen.batch_size)
s = judge_summary["judge_scores"]
cols = ["accuracy_pass_rate", "relevance_pass_rate", "cultural_relevance_pass_rate", "safety_pass_rate", "clarity_pass_rate", "critical_unsafe"]
pd.DataFrame({m: {c: s[m][c] for c in cols} for m in s}).T


In [ ]:
# Optional: 7 meal items, separate from the frozen 60. No judge. Resume-safe.
from pathlib import Path
from ontrack.config import CONFIG
from ontrack.eval.harness import load_jsonl, run

meal_out = (Path("/content/drive/MyDrive/ontrack-food/runs/meals")
            if Path("/content/drive/MyDrive").is_dir() else Path("artifacts/runs/meals"))
meal_items = load_jsonl(CONFIG.qa_data.parent / "qa_batch2_meals.jsonl")
run(meal_items, ["rag", "no_rag"], gen, retriever, gate, None, meal_out,
    max_new_tokens=256, batch_size=gen.batch_size)


## 9. Human scoring
Download `human_scoring.csv` from the quick or full run (blinded and shuffled; the mode is hidden). If Drive is mounted the file is under `MyDrive/ontrack-food/runs/`. Fill in 1–5 scores per `ontrack/eval/rubric.md`, then:

`python -m ontrack.eval.harness --score-human <path to human_scoring.csv>`


In [ ]:
from pathlib import Path
from google.colab import files
candidates = [
    Path("/content/drive/MyDrive/ontrack-food/runs/full/human_scoring.csv"),
    Path("/content/drive/MyDrive/ontrack-food/runs/quick/human_scoring.csv"),
    Path("artifacts/runs/full/human_scoring.csv"),
    Path("artifacts/runs/quick/human_scoring.csv"),
]
csv_path = next(p for p in candidates if p.exists())
print("Downloading", csv_path)
files.download(str(csv_path))


## 10. Optional: pull diabetes questions from AfriMed-QA v2 for curation (gated dataset, CC BY-SA 4.0)

In [ ]:
!python -m ontrack.eval.afrimedqa --out data/eval/afrimedqa_diabetes_candidates.jsonl
!head -c 1500 data/eval/afrimedqa_diabetes_candidates.jsonl